# **data extraction**
this section covers:
- HDB resale prices
- onemap API auth and retrieveThemes

steps taken:

- extract data from data.gov (no auth necessary)
- authenticate onemap api
- extract data from onemap api
- save raw page-level JSON responses for traceability
- aggregate and save outputs as csv for ease of use

raw outputs go to `data/raw/` in whatever formats they come in, for manual handling and specific cleaning.

In [8]:
import os
import requests
import json
from pathlib import Path
from datetime import datetime

CACHE_FILE = Path("./data/.onemap_token_cache.json")

if CACHE_FILE.exists():
    try:
        with open(CACHE_FILE, "r") as f:
            cache = json.load(f)
        if cache["expiry_timestamp"] > datetime.now().timestamp():
            print("existing onemap token has not expired, using cached token")
            create = False
        else:
            print("Cached token expired, requesting new one...")
            create = True
    except Exception as e:
        print("Failed to read cache, requesting new token...", e)

if create:
    # Request new token
    url = "https://www.onemap.gov.sg/api/auth/post/getToken"
    payload = {
        "email": os.environ["ONEMAP_EMAIL"],
        "password": os.environ["ONEMAP_PASSWORD"]
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()
    data = response.json()

    # Save to cache
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, "w") as f:
        json.dump({
            "access_token": data["access_token"],
            "expiry_timestamp": int(data["expiry_timestamp"])
        }, f)

    print("Saved new OneMap token to cache")

Cached token expired, requesting new one...
Saved new OneMap token to cache


In [ ]:
"""
querying for hdb resale prices and other data from data.gov.sg

Resale HDB prices (transaction-level) - d_8b84c4ee58e3cfc0ece0d773c8ca6abc
HDB BTO prices (aggregate-level) -      d_67966e5fd5dce14cf9fa5f0bc5164faf
"""

from pathlib import Path
import requests
import pandas as pd

# Configuration + init
DATASTORE_URL = "https://data.gov.sg/api/action/datastore_search"
LIMIT = 10000

DATASTORE_RESOURCE_IDS = [
    "d_8b84c4ee58e3cfc0ece0d773c8ca6abc",  # Resale prices
    "d_67966e5fd5dce14cf9fa5f0bc5164faf",  # BTO prices (aggregated)
]

DATASTORE_RESOURCE_ID_TO_NAME = {
    "d_8b84c4ee58e3cfc0ece0d773c8ca6abc": "hdb_resale_prices",
    "d_67966e5fd5dce14cf9fa5f0bc5164faf": "hdb_bto_prices_aggregated",
}

# Output directories
BASE_DIR = Path("./data")
RUN_DIR = BASE_DIR / "raw"
JSON_DIR = RUN_DIR / "json_api"
CSV_DIR = RUN_DIR / "csv_api"

for d in [JSON_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Query + Save with pagination
for resource_id in DATASTORE_RESOURCE_IDS:
    name = DATASTORE_RESOURCE_ID_TO_NAME[resource_id]
    offset = 0
    page = 1
    all_records = []

    while True:
        url = f"{DATASTORE_URL}?resource_id={resource_id}&limit={LIMIT}&offset={offset}"
        response = requests.get(url)
        data = response.json()
        
        records = data.get("result", {}).get("records", [])
        if not records:
            break  # no more data

        # Save JSON page
        json_path = JSON_DIR / f"{name}_{page:02d}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            f.write(response.text)

        print(f"✅ Page {page} | {len(records)} rows saved to {json_path}")

        # Append records for merged CSV
        all_records.extend(records)

        # Prepare next loop
        offset += LIMIT
        page += 1

    # Save combined CSV (all pages together)
    if all_records:
        df_all = pd.DataFrame(all_records)
        combined_csv_path = CSV_DIR / f"{name}.csv"
        df_all.to_csv(combined_csv_path, index=False)
        print(f"📦 Combined {len(df_all)} rows into {combined_csv_path}")

✅ Page 1 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_01.json
✅ Page 2 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_02.json
✅ Page 3 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_03.json
✅ Page 4 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_04.json
✅ Page 5 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_05.json
✅ Page 6 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_06.json
✅ Page 7 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_07.json
✅ Page 8 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_08.json
✅ Page 9 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_09.json
✅ Page 10 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_10.json
✅ Page 11 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_11.json
✅ Page 12 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_12.json
✅ Page 13 | 10000 rows saved to data/raw/json_api/hdb_resale_prices_13.json
✅ Page 14 | 10000 row

In [21]:
"""
querying for amenity data from onemap API

there are more, but these are the ones i query for:
sportsg_sport_facilities: Sport facilities
ssot_hawkercentres: Hawker centres
nationalparks: National parks
"""

import requests
import json
from pathlib import Path
from datetime import datetime

ONEMAP_RESOURCE_IDS = [
    "sportsg_sport_facilities",  # Sport facilities
    "ssot_hawkercentres",        # Hawker centres
    "nationalparks"              # National parks
]

ONEMAP_RESOURCE_ID_TO_NAME = {
    "sportsg_sport_facilities": "sport_facilities",
    "ssot_hawkercentres": "hawker_centres",
    "nationalparks": "national_parks"
}

CACHE_FILE = Path("./data/.onemap_token_cache.json")
RAW_DIR = Path("./data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)  # ensure ./data/raw exists

# read token cache file, check that it is not expired
with open(CACHE_FILE, "r") as f:
    cache = json.load(f)
if cache["expiry_timestamp"] < datetime.now().timestamp():
    raise ValueError("Existing OneMap token has expired, request a new authorization token")

ONEMAP_TOKEN = cache["access_token"]

ONEMAP_URL = "https://www.onemap.gov.sg/api/public/themesvc/retrieveTheme?queryName="

headers = {"Authorization": ONEMAP_TOKEN}

# Loop over resource IDs
for resource_id in ONEMAP_RESOURCE_IDS:
    url = ONEMAP_URL + resource_id
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # raise error if request fails
    
    data = response.json()
    
    # map to friendly name
    name = ONEMAP_RESOURCE_ID_TO_NAME[resource_id]
    
    # save to ./data/raw/<name>_raw.json
    out_path = RAW_DIR / f"{name}_raw.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {resource_id} → {out_path}")

Saved sportsg_sport_facilities → data/raw/sport_facilities_raw.json
Saved ssot_hawkercentres → data/raw/hawker_centres_raw.json
Saved nationalparks → data/raw/national_parks_raw.json


In [ ]:
"""
perform initial null checks for HDB resale price data + amenity data
"""

import pandas as pd
from pathlib import Path

CSV_DIR = Path("./data/raw/csv_api")
RAW_DIR = Path("./data/raw")

# load HDB resale price data
resource_to_df = {}
for csv_file in CSV_DIR.glob("*.csv"):
    df = pd.read_csv(csv_file)
    resource_name = csv_file.stem  # get the name without extension
    resource_to_df[resource_name] = df
    
# check for null values
null_checks = {}
for resource_name, df in resource_to_df.items():
    null_counts = df.isnull().sum()
    null_checks[resource_name] = null_counts[null_counts > 0]  # only keep columns with nulls
    
# print null checks
for resource_name, null_counts in null_checks.items():
    if not null_counts.empty:
        print(f"Null values in {resource_name}:")
        print(null_counts)
    else:
        print(f"No null values in {resource_name}")

# do the same for amenity data

# load amenity data
amenity_data = {}
for json_file in RAW_DIR.glob("*_raw.json"):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        amenity_name = json_file.stem.replace("_raw", "")  # remove _raw suffix
        amenity_data[amenity_name] = data

# check for null values in amenity data
amenity_null_checks = {}
for amenity_name, data in amenity_data.items():
    null_counts = {}
    for key, value in data.items():
        if isinstance(value, list):
            null_counts[key] = 0 if value else 1
        elif value is None:
            null_counts[key] = 1
        else:
            null_counts[key] = 0
    amenity_null_checks[amenity_name] = {k: v for k, v in null_counts.items() if v > 0}
    
# print amenity null checks
for amenity_name, null_counts in amenity_null_checks.items():
    if null_counts:
        print(f"Null values in {amenity_name}:")
        for key, count in null_counts.items():
            print(f"  {key}: {count}")
    else:
        print(f"No null values in {amenity_name}")

No null values in hdb_resale_prices
No null values in hdb_bto_prices_aggregated
No null values in national_parks
No null values in activesg
No null values in sport_facilities
No null values in hawker_centres


# **data manipulation and transformation**

due to the lack of APIs and also minute differences in data formatting, each file is processed slightly differently from each separate source.

manual transformations applied and documented within this section, for data which we will use in the below analyses.

a note on HDB geocoordinate data:
- geocoordinates are not provided in original dataset from data.gov.
- i use onemap search api to retrieve the data, and did so separately in another script.
- to validate, i implement my own validation strategy by using 2019 planning area boundary data by checking a particular coordinate lies in a town boundary, under the field `valid_in_town_area`.

store cleaned and validated data in `data/prod`

In [ ]:
"""
HDB resale geocoordinate data validation

to validate the geocoordinates of HDB resale transactions, we check if each point (LONGITUDE, LATITUDE) lies within the specified town boundary.
this is done using the 2019 planning area boundary data, which provides polygons for each town.

0 = does not lie within town specified boundary, 
1 = lies within town boundary, 
NaN = non-exact mapping to town between HDB resale dataset and planning area boundaries.
"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

PROD_DIR = "./data/prod/"
STG_DIR = "./data/stg/"

def validate_points_in_town(hdb_data: pd.DataFrame, geojson_path: str) -> pd.DataFrame:
    """
    Validate whether each HDB point (LONGITUDE, LATITUDE) falls within its corresponding town's polygon.
    
    Args:
        hdb_data (pd.DataFrame): DataFrame containing HDB data with LONGITUDE, LATITUDE, and town columns
        geojson_path (str): Path to the GeoJSON file containing town polygon data
        
    Returns:
        pd.DataFrame: Original DataFrame with new column 'valid_in_town_area' (1 if valid, 0 if invalid, None if town not found)
    """
    # Read GeoJSON file
    gdf = gpd.read_file(geojson_path)

    # Convert town names to uppercase for matching
    gdf["pln_area_n"] = gdf["pln_area_n"].str.upper()

    # Create a copy of the input DataFrame
    result_df = hdb_data.copy()
    result_df["valid_in_town_area"] = None

    # Create points for each HDB entry
    points = [Point(lon, lat) for lon, lat in zip(hdb_data["LONGITUDE"], hdb_data["LATITUDE"])]
    points_gdf = gpd.GeoDataFrame(
        {"geometry": points, "town": hdb_data["town"]},
        crs="EPSG:4326"
    )

    # Process each town separately
    for town in result_df["town"].unique():
        try:
            # Get the polygon for this town
            town_polygon = gdf[gdf["pln_area_n"] == town].iloc[0].geometry

            # Get points for this town
            town_points = points_gdf[points_gdf["town"] == town]
            
            # Find which points are within the polygon
            town_mask = result_df["town"] == town
            result_df.loc[town_mask, "valid_in_town_area"] = town_points.geometry.apply(lambda x: 1 if x.within(town_polygon) else 0)
            
        except IndexError:
            # Town not found in GeoJSON
            print(f"Warning: Town '{town}' not found in GeoJSON data")
            continue
        except Exception as e:
            print(f"Error processing town '{town}': {str(e)}")
            continue

    return result_df

# After your enrich stage
enriched_df = pd.read_csv(STG_DIR + "hdb_resale_prices_enriched_cleaned.csv")

# Run validation
validated_df = validate_points_in_town(
    enriched_df, 
    PROD_DIR + "planningarea_data_clean.geojson"
)

# Save validated data
validated_df.to_csv(STG_DIR + "hdb_resale_prices_enriched_validated.csv", index=False)

In [31]:
"""
HDB resale price data processing
manual creation of remaining_lease_numeric and address columns

create remaining_lease_numeric column from remaining_lease string
- converts "99 years 6 months" to 99.5
- handles missing values by returning None if remaining_lease is NaN
- applies to each row in the DataFrame

create address column, which is a concatenation of block, street_name
- e.g. "123 Bukit Timah Road"
- handles missing values by returning None if either block or street_name is NaN
- applies to each row in the DataFrame

returns csv
"""

import re
import pandas as pd

STG_DIR = './data/stg/'
PROD_DIR = './data/prod/'

# Load your CSV file
filename = 'hdb_resale_prices_enriched_validated.csv'  # Replace (file_name) with your actual file name
df = pd.read_csv(STG_DIR + filename)

def lease_to_years(lease_str):
    if pd.isnull(lease_str):
        return None
    # Extract years and months using regex
    years = 0
    months = 0
    years_match = re.search(r'(\d+)\s*year', lease_str)
    months_match = re.search(r'(\d+)\s*month', lease_str)
    if years_match:
        years = int(years_match.group(1))
    if months_match:
        months = int(months_match.group(1))
    return round(years + (months / 12),2)

# function to create address column, which is a concatenation of block, street_name
def create_address(row):
    block = row['block']
    street_name = row['street_name']
    if pd.isnull(block) or pd.isnull(street_name):
        return None
    return f"{block} {street_name}"

df['remaining_lease_numeric'] = df['remaining_lease'].apply(lease_to_years)
df['address'] = df.apply(create_address, axis=1)

final_filename = "hdb_resale_prices_final.csv"

# Save the updated DataFrame to csv in prod dir
df.to_csv(PROD_DIR + final_filename, index=False)
print(f"HDB resale prices data processing complete. Output saved to: {PROD_DIR + final_filename}")

HDB resale prices data processing complete. Output saved to: ./data/prod/hdb_resale_prices_final.csv


In [16]:
"""
Bus stop first-step data processing 
manually convert shp to geojson using geopandas
"""
import geopandas as gpd

RAW_DIR = './data/raw/'

gdf = gpd.read_file(RAW_DIR + "BusStopLocation_Apr2025/BusStop.shp")
gdf.to_file(RAW_DIR + "busstops.geojson", driver="GeoJSON")

gdf
print("first-layer bus stop data processing complete. Output saved to:", RAW_DIR + "busstops.geojson")

first-layer bus stop data processing complete. Output saved to: ./data/raw/busstops.geojson


In [17]:
"""
Bus stop second-step data processing
manual conversion of bus stop locations from SVY21 to WGS84 using pyproj

input: geojson file with bus stop locations in SVY21 coordinates
returns CSV file with bus stop locations in WGS84 coordinates in data/prod
"""

import json
import csv
from pyproj import Transformer

RAW_DIR = './data/raw/'
PROD_DIR = './data/prod/'

# SVY21 (EPSG:3414) to WGS84 (EPSG:4326)
transformer = Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True)

with open(RAW_DIR + 'busstops_raw.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

with open(PROD_DIR + 'busstops_wgs84.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['BUS_STOP_N', 'LOC_DESC', 'LATITUDE', 'LONGITUDE'])
    for feature in data['features']:
        props = feature['properties']
        coords = feature['geometry']['coordinates']
        easting, northing = coords[0], coords[1]
        lon, lat = transformer.transform(easting, northing)
        writer.writerow([props.get('BUS_STOP_N', ''), props.get('LOC_DESC', ''), lat, lon])

print("second-layer bus stop data processing complete. Output saved to:", PROD_DIR + 'busstops_wgs84.csv')

second-layer bus stop data processing complete. Output saved to: ./data/prod/busstops_wgs84.csv


In [11]:
"""
MRT and LRT exit data processing
manual extraction of station name and exit code from Description field in geojson

input: geojson file with MRT/LRT exits and their geocoordinates
returns CSV file with station name, exit code, longitude, and latitude in data/prod
"""

import json
import csv
import re

RAW_DIR = './data/raw/'
PROD_DIR = './data/prod/'

def extract_station_info(description):
    # Extract station name and exit code using regex
    station_match = re.search(r'<th>STATION_NA<\/th>\s*<td>(.*?)<\/td>', description)
    exit_match = re.search(r'<th>EXIT_CODE<\/th>\s*<td>(.*?)<\/td>', description)
    station_name = station_match.group(1) if station_match else ''
    exit_code = exit_match.group(1) if exit_match else ''
    return station_name, exit_code

with open(RAW_DIR + 'lrt_and_mrt_exits_geocoordinates_raw.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

with open(PROD_DIR + 'lrt_and_mrt_exits_geocoordinates.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['station_name', 'exit_code', 'longitude', 'latitude'])
    for feature in data['features']:
        props = feature['properties']
        geom = feature['geometry']
        description = props.get('Description', '')
        station_name, exit_code = extract_station_info(description)
        longitude, latitude, *_ = geom['coordinates']
        writer.writerow([station_name, exit_code, longitude, latitude])

print("MRT and LRT exit data processing complete. Output saved to:", PROD_DIR + 'lrt_and_mrt_exits_geocoordinates.csv')

MRT and LRT exit data processing complete. Output saved to: ./data/prod/lrt_and_mrt_exits_geocoordinates.csv


In [7]:
"""
2019 planning area data processing
manual conversion of planning area data from JSON to GeoJSON format

input: JSON file with planning area data
returns GeoJSON file with planning area polygons in data/prod
"""

import json
from shapely.geometry import shape, mapping
from shapely.validation import make_valid

RAW_DIR = './data/raw/'
PROD_DIR = './data/prod/'

# Input & output paths
input_path = RAW_DIR + "planningarea_raw.geojson"
output_path = PROD_DIR + "planningarea_polygons.geojson"

# Step 1: Read raw file
with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Step 2: Try to parse JSON
try:
    data = json.loads(raw_text)
except json.JSONDecodeError as e:
    raise ValueError(f"Invalid JSON: {e}")

# Step 3: Transform data into proper GeoJSON features
features = []
for entry in data:
    # Parse the nested GeoJSON string into a proper object
    try:
        geometry = json.loads(entry["geojson"])
    except (KeyError, json.JSONDecodeError) as e:
        print(f"⚠ Error parsing geometry: {e}")
        continue
    
    # create proper GeoJSON feature
    feature = {
        "type": "Feature",
        "properties": {
            "pln_area_n": entry.get("pln_area_n", "")
        },
        "geometry": geometry
    }
    
    try:
        geom = shape(geometry)
        if not geom.is_valid:
            print(f"fixing invalid geometry for {entry.get('pln_area_n', 'Unknown area')}...")
            geom = make_valid(geom)
            feature["geometry"] = mapping(geom)
        features.append(feature)
    except Exception as e:
        print(f"error validating geometry for {entry.get('pln_area_n', 'Unknown area')}: {e}")
        continue

# Create the final GeoJSON structure
cleaned_geojson = {
    "type": "FeatureCollection",
    "features": features
}

# Save cleaned file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_geojson, f, ensure_ascii=False, indent=2)

print(f"Cleaned & fixed GeoJSON saved to {output_path}")
print(f"Total features processed: {len(features)}")

⚠ Fixing invalid geometry for BEDOK...
⚠ Fixing invalid geometry for BUKIT MERAH...
⚠ Fixing invalid geometry for NORTH-EASTERN ISLANDS...
⚠ Fixing invalid geometry for SOUTHERN ISLANDS...
⚠ Fixing invalid geometry for JURONG EAST...
Cleaned & fixed GeoJSON saved to ./data/prod/planningarea_polygons.geojson
Total features processed: 55


In [ ]:
"""
parks, activesg and hawkercentre data processing
manual conversion of planning area data from JSON to GeoJSON format

input: JSON file with planning area data
returns GeoJSON file with planning area polygons in data/prod
"""

import json
import pandas as pd

def convert_parks_json_to_csv(json_file_path: str, output_file_path: str) -> None:
    """
    Convert parks JSON data to CSV format.
    
    Args:
        json_file_path (str): Path to the JSON file
        output_file_path (str): Path to save the CSV file
        
    Example:
        convert_parks_json_to_csv(
            "data/raw/parks_raw.json",
            "data/cleaned/parks_data.csv"
        )
    """
    # Read JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Extract metadata from first entry
    metadata = data['SrchResults'][0]
    
    # Extract park entries (skip the first metadata entry)
    parks = data['SrchResults'][1:]
    
    # Convert to DataFrame
    df = pd.DataFrame(parks)
    
    # Split LatLng into separate columns
    df[['LATITUDE', 'LONGITUDE']] = df['LatLng'].str.split(',', expand=True)
    
    # Convert coordinates to float
    df['LATITUDE'] = pd.to_numeric(df['LATITUDE'])
    df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'])
    
    # Add metadata columns
    df['Theme_Name'] = metadata['Theme_Name']
    df['Category'] = metadata['Category']
    df['Owner'] = metadata['Owner']
    df['DateTime'] = metadata['DateTime']
    df['Published_Date'] = metadata['Published_Date']
    
    # Reorder columns
    column_order = [
        'NAME', 'Theme_Name', 'Category', 'Owner',
        'Type', 'LATITUDE', 'LONGITUDE',
        'ICON_NAME', 'DateTime', 'Published_Date'
    ]
    df = df[column_order]
    
    # Save to CSV
    df.to_csv(output_file_path, index=False)
    print(f"Converted {len(df)} parks to CSV: {output_file_path}")
    print(f"Columns: {', '.join(df.columns)}")

RAW_DIR = './data/raw/'
PROD_DIR = './data/prod/'

files = [
    "parks",
    "activesg",
    "hawkercentres"
]

for filename in files:
    json_path = f"{RAW_DIR}{filename}_raw.json"
    csv_path = f"{PROD_DIR}{filename}_data.csv"

    convert_parks_json_to_csv(json_path, csv_path)

Converted 447 parks to CSV: ./data/prod/parks_data.csv
Columns: NAME, Theme_Name, Category, Owner, Type, LATITUDE, LONGITUDE, ICON_NAME, DateTime, Published_Date
Converted 45 parks to CSV: ./data/prod/activesg_data.csv
Columns: NAME, Theme_Name, Category, Owner, Type, LATITUDE, LONGITUDE, ICON_NAME, DateTime, Published_Date
Converted 129 parks to CSV: ./data/prod/hawkercentres_data.csv
Columns: NAME, Theme_Name, Category, Owner, Type, LATITUDE, LONGITUDE, ICON_NAME, DateTime, Published_Date


# **aggregated data visualization using kepler.gl**

in this section, we aim to visualize the data to spot any broad trends that might be worth zoning in on

In [1]:
# dependencies
import pandas as pd
from keplergl import KeplerGl
import json

# loading of data into memory
PROD_DIR = './data/prod/'
CONFIG_DIR = './keplergl/configs/'

# loading saved keplergl config
with open(CONFIG_DIR + "keplergl_config.json", "r") as f:
    config = json.load(f)

with open(CONFIG_DIR + "lw_keplergl_config.json", "r") as f:
    lw_config = json.load(f)

with open(PROD_DIR + "planningarea_polygons.geojson") as f:
    planning_area_geojson = json.load(f)

hdb_df = pd.read_csv(PROD_DIR + "hdb_resale_prices_final.csv")
busstops_data = pd.read_csv(PROD_DIR + "busstops_wgs84.csv")
resale_hdbs_data = pd.read_csv(PROD_DIR + "hdb_resale_prices_final.csv")
mrt_lrt_exits = pd.read_csv(PROD_DIR + "lrt_and_mrt_exits_geocoordinates.csv")

print("load complete")

load complete


In [2]:
runtype = "w"

if runtype == "lw":
    hdbdf_subset = hdb_df[["town", "resale_price", "remaining_lease_numeric", "address", "LONGITUDE", "LATITUDE"]].copy()
    
    map_1 = KeplerGl(height=800, config=lw_config, data={
        "resale_hdbs": hdbdf_subset,
        "mrt_lrt": mrt_lrt_exits,
        "planning_area": planning_area_geojson
    })

else:
    data = {
        "resale_hdbs": hdb_df,
        "mrt_lrt": mrt_lrt_exits,
        "bus_stops": busstops_data,
        "planning_area": planning_area_geojson
    }

    map_1 = KeplerGl(height=800, config=config, data=data)

map_1

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


/opt/anaconda3/lib/python3.12/site-packages/jupyter_client/session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [{'dataId': ['resale_hdbs'], 'id': 'e3phw…

In [ ]:
runtype == "lw"

MAPS_DIR = './keplergl/maps/'

# save current map config to JSON
if runtype == "lw":
    with open("lw_keplergl_config.json", "w") as f:
        json.dump(map_1.config, f, indent=4)
else:
    with open("keplergl_config.json", "w") as f:
        json.dump(map_1.config, f, indent=4)

map_1.save_to_html(file_name=MAPS_DIR + ('lw_keplergl_map.html' if runtype == "lw" else 'keplergl_map.html'), read_only=True)

Map saved to ./keplergl/maps/keplergl_map.html!


# **proximity analysis for HDB resale transactions**

for each HDB resale transaction, we search for the name and distance to the nearest amenity below:
1. MRT Stations (excluding LRT stations)
2. Hawker Centres
3. ActiveSG Facilities 
4. Parks

In [ ]:
import pandas as pd
import numpy as np

# Load all required datasets
PROD_DIR = './data/prod/'

# Load HDB resale data
hdb_df = pd.read_csv(PROD_DIR + 'hdb_resale_prices_final.csv')

# Load amenity datasets
mrt_df = pd.read_csv(PROD_DIR + 'lrt_and_mrt_exits_geocoordinates.csv')
hawker_df = pd.read_csv(PROD_DIR + 'hawkercentres_data.csv')
activesg_df = pd.read_csv(PROD_DIR + 'activesg_data.csv')
parks_df = pd.read_csv(PROD_DIR + 'parks_data.csv')
busstops_df = pd.read_csv(PROD_DIR + 'busstops_wgs84.csv')

mrt_df.rename(columns={
    'latitude': 'LATITUDE',
    'longitude': 'LONGITUDE',
}, inplace=True)

def filter_mrt_stations(df: pd.DataFrame) -> pd.DataFrame:
    """filter MRT stations and exclude LRT stations"""
    for row in df.itertuples():
        if 'LRT' in row.station_name:
            df.drop(row.Index, inplace=True)
    return df

# Filter for MRT stations only
mrt_stations_df = filter_mrt_stations(mrt_df)

print("Datasets loaded successfully:")
print(f"HDB Transactions: {len(hdb_df):,}")
print(f"MRT Stations: {len(mrt_stations_df):,}")
print(f"Bus Stops: {len(busstops_df):,}")
print(f"Hawker Centres: {len(hawker_df):,}")
print(f"ActiveSG Facilities: {len(activesg_df):,}")
print(f"Parks: {len(parks_df):,}")

clean = True
# check if any NAs exist in the coordinates.
for df in [hdb_df, mrt_stations_df, hawker_df, activesg_df, parks_df, busstops_df]:
    if df[['LATITUDE', 'LONGITUDE']].isna().any().any():
        print(f"Found NaN coordinates in {df.name if hasattr(df, 'name') else 'DataFrame'}")
        display(df[df['LATITUDE'].isna() | df['LONGITUDE'].isna()])
        clean = False

if clean:
    print("\nall datasets have valid coordinates")

Datasets loaded successfully:
HDB Transactions: 213,342
MRT Stations: 491
Bus Stops: 5,170
Hawker Centres: 129
ActiveSG Facilities: 45
Parks: 447

all datasets have valid coordinates


In [29]:
from sklearn.neighbors import BallTree
import numpy as np

EARTH_RADIUS_M = 6371000

def process_nearest_amenity_batches(
    hdb_df,
    amenity_df,
    amenity_name_col,
    amenity_label=None,
    lat_col='LATITUDE',
    lon_col='LONGITUDE'
):
    """
    Find nearest amenity for every HDB record using BallTree.

    Parameters
    ----------
    hdb_df : pd.DataFrame
        HDB dataframe containing LATITUDE and LONGITUDE.

    amenity_df : pd.DataFrame
        Amenity dataframe containing LATITUDE and LONGITUDE.

    amenity_name_col : str
        Column containing amenity names.

    amenity_label : str, optional
        For logging/progress messages.

    lat_col : str
        Latitude column name.

    lon_col : str
        Longitude column name.

    Returns
    -------
    nearest_names : np.ndarray
    nearest_distances : np.ndarray
        Distances in meters.
    """

    if amenity_label:
        print(f"Processing nearest {amenity_label}...")

    # Amenity coordinates in radians
    amenity_coords = np.radians(
        amenity_df[[lat_col, lon_col]].values
    )

    # Build BallTree
    tree = BallTree(
        amenity_coords,
        metric='haversine'
    )

    # HDB coordinates in radians
    hdb_coords = np.radians(
        hdb_df[[lat_col, lon_col]].values
    )

    # Query nearest amenity
    distances, indices = tree.query(
        hdb_coords,
        k=1
    )

    nearest_names = (
        amenity_df.iloc[indices.flatten()][amenity_name_col]
        .values
    )

    nearest_distances = (
        distances.flatten() * EARTH_RADIUS_M
    )

    return nearest_names, nearest_distances

In [30]:
# MRT stations
mrt_names, mrt_distances = process_nearest_amenity_batches(
    hdb_df,
    mrt_stations_df,
    'station_name',
    'MRT stations'
)

hdb_df['nearest_mrt_exit'] = mrt_names
hdb_df['distance_to_mrt'] = np.round(mrt_distances, 2)


# Hawker centres
hawker_names, hawker_distances = process_nearest_amenity_batches(
    hdb_df,
    hawker_df,
    'NAME',
    'hawker centres'
)

hdb_df['nearest_hawker'] = hawker_names
hdb_df['distance_to_hawker'] = np.round(hawker_distances, 2)


# ActiveSG facilities
activesg_names, activesg_distances = process_nearest_amenity_batches(
    hdb_df,
    activesg_df,
    'NAME',
    'ActiveSG facilities'
)

hdb_df['nearest_activesg'] = activesg_names
hdb_df['distance_to_activesg'] = np.round(activesg_distances, 2)


# Parks
park_names, park_distances = process_nearest_amenity_batches(
    hdb_df,
    parks_df,
    'NAME',
    'parks'
)

hdb_df['nearest_park'] = park_names
hdb_df['distance_to_park'] = np.round(park_distances, 2)

# Bus Stops
bus_names, bus_distances = process_nearest_amenity_batches(
    hdb_df,
    busstops_df,
    'LOC_DESC',
    'bus stops'
)

hdb_df['nearest_bus_stop'] = bus_names
hdb_df['distance_to_bus_stop'] = np.round(bus_distances, 2)

# filter out the unvalidated data
hdb_df = hdb_df[hdb_df['valid_in_town_area'] == 1]

# hdb_df.to_csv(PROD_DIR + 'hdb_resale_prices_final_validated.csv', index=False)

Processing nearest MRT stations...
Processing nearest hawker centres...
Processing nearest ActiveSG facilities...
Processing nearest parks...
Processing nearest bus stops...
